# Task Ledger: Workflow-Fix + Smoke-Test

Original request: https://github.com/karlokarate/kannalles1/actions/runs/29194661748 erneuter fail. kannst du bitte mal alles validiert fixen inkl smoketest, des wf um weitere fehler auszuschließen?

Scope:
- Root-Cause Analyse des fehlgeschlagenen GH Actions Runs 29194661748
- Minimal-invasive Fixes in Workflow/Code für reproduzierbar grünen Prepare-Job
- Lokaler Smoke-Test der Workflow-Pipelinepfade

Non-goals:
- Keine funktionalen Produktfeatures
- Keine Änderungen an Secrets/Deploy-Credentials

Allowed files:
- .github/workflows/build-deploy-pages.yml
- src/App.tsx
- scripts/* (nur falls zwingend)
- .codex/task-ledgers/* und docs/v3/archive/handovers/*

Forbidden files:
- Unrelated product logic outside build/deploy failure path

Expected behavior:
- Workflow bricht nicht mehr im Prepare-Job durch den bekannten Fehlerpfad
- Typecheck bleibt grün
- Final Pages contract checks im Smoke-Test bestehen

Validation gates:
- npm --prefix /workspaces/kannalles1 run typecheck
- npm --prefix /workspaces/kannalles1 run check:workflow
- env VITE_DATA_GATEWAY_URL=https://kannalles1.vercel.app ONE_CLICK_SKIP_BROWSER=1 bash /workspaces/kannalles1/scripts/one-click-prepare.sh

Execution rule:
- Kleinste sichere Änderung, dann reproduzierbare lokale Verifikation, dann Push.

## Checklist (JSON)
```json
{
  "steps": [
    {"id": "s1", "title": "Run-Root-Cause verifizieren", "status": "done"},
    {"id": "s2", "title": "Workflow/Code minimal fixen", "status": "in_progress"},
    {"id": "s3", "title": "Smoke-Test lokal", "status": "pending"},
    {"id": "s4", "title": "Push + Handover", "status": "pending"}
  ],
  "touch_budget": {"max_files": 4},
  "validation_gates": ["typecheck", "check:workflow", "one-click-prepare smoke"],
  "findings_backlog": [],
  "handover": {"required": true, "path": "docs/v3/archive/handovers/"}
}
```

## STEP-0-AUDIT-ARTIFACT
```json
{
  "run_id": 29194661748,
  "failing_step": "Run serial generator, quality and release pipeline",
  "root_cause": "TS2304 in src/App.tsx: searchBlockedRemainingMs undefined after patch application path",
  "secondary_failure": "verify-pages-build fails because fallback publish-site lacks configured gateway string when source candidate disqualified",
  "relevant_files": [
    "src/App.tsx",
    ".github/workflows/build-deploy-pages.yml",
    "scripts/one-click-prepare.sh",
    "scripts/verify-pages-build.mjs"
  ],
  "existing_tests": [
    "npm run typecheck",
    "npm run check:workflow",
    "bash scripts/one-click-prepare.sh (smoke)"
  ],
  "risk_points": [
    "Workflow patch step mutates source before typecheck",
    "no-cooldown patch may drift versus current App.tsx"
  ],
  "out_of_scope_findings": [
    "AGENTS.md and contracts/v3 ledger contract files not present in this repository layout"
  ],
  "recommended_touch_budget": 2
}
```

## Validation Evidence
- `npm --prefix /workspaces/kannalles1 run check:workflow` => pass
- `npm --prefix /workspaces/kannalles1 run typecheck` => pass
- `CI=true VITE_DATA_GATEWAY_URL=https://kannalles1.vercel.app ONE_CLICK_SKIP_BROWSER=1 bash scripts/one-click-prepare.sh` => pass (full prepare smoke, browser gate intentionally skipped)

## Changed Files
- `.github/workflows/build-deploy-pages.yml`
- `.codex/task-ledgers/2026-07-12_workflow-smoketest-fix.ipynb`

## FINAL-RECONCILIATION
- Root cause reproduced and confirmed from run `29194661748`: CI patch application mutated code and introduced TypeScript failure before publish selection.
- Resolution implemented: workflow no longer mutates source via hotfix patches; it only verifies patch assets are syntactically valid.
- Local smoke-test validated end-to-end prepare pipeline with gateway env configured.
- Residual risk: Playwright/browser gates were skipped in smoke by design (`ONE_CLICK_SKIP_BROWSER=1`), matching a fast pre-merge smoke objective.